In [4]:
import os
import sys

import math
import time
import datetime
import numpy as np
from numpy.lib.stride_tricks import sliding_window_view
import torch
from torch.utils.data import Dataset
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torch.optim.lr_scheduler import CosineAnnealingLR
from MnM import MnM
from torchinfo import summary
import torchprofile

# from YourDataset import YourDataset  # Import your custom dataset here
from tqdm import tqdm
from torch.cuda.amp import autocast, GradScaler

import matplotlib.pyplot as plt

import pickle
torch.manual_seed(23)

DTYPE = torch.float32

scaler = GradScaler()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

import matplotlib
matplotlib.rcParams["figure.dpi"] = 200
plt.rcParams["font.family"] = "serif"

import scipy.stats as stats

Using device: cuda


In [5]:
# Define your custom loss function here
class CustomLoss(nn.Module):
    def __init__(self, Par):
        super(CustomLoss, self).__init__()

    def forward(self, y_pred, y_true, Par):
        BS, NF, LF, NX, NY = y_true.shape
        y_pred = (y_pred - Par['out_shift_loss'])/Par['out_scale_loss']
        y_true = (y_true - Par['out_shift_loss'])/Par['out_scale_loss']
        # loss = torch.mean( ((y_pred-y_true))**2 )/torch.mean((y_true)**2 )
        loss = 1 / (BS*NX*NY) * torch.sum( (y_true - y_pred)**2 )

        return loss

class YourDataset(Dataset):
    def __init__(self, x, y, transform=None):
        self.x = x
        self.y = y
        self.transform = transform

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        x_sample = self.x[idx]
        y_sample = self.y[idx]

        if self.transform:
            x_sample, y_sample = self.transform(x_sample, y_sample)

        return x_sample, y_sample


def preprocess_train(traj, Par):
    # traj - [bs, nt, nx, ny, nf]

    subsample_t = Par['subsample_t']
    x = sliding_window_view(traj[:,:-(Par['lf']),:,:], window_shape=Par['lb'], axis=1 ).transpose(0,1,5,2,3,4)[:,::subsample_t].reshape(-1,Par['lb'],Par['nx'], Par['ny'],Par['nf'])
    y = sliding_window_view(traj[:,Par['lb']:,:,:], window_shape=Par['lf'], axis=1 ).transpose(0,1,5,2,3,4)[:,::subsample_t].reshape(-1,Par['lf'],Par['nx'], Par['ny'],Par['nf'])

    x = x.transpose(0,4,1,2,3) #[bs, nf, lb,nx,ny]
    y = y.transpose(0,4,1,2,3) #[bs, nf, lf,nx,ny]

    print('x: ', x.shape)
    print('y: ', y.shape)
    print()
    return x.astype(np.float32), y.astype(np.float32)


def preprocess(traj, Par):
    # traj - [bs , nt, nx , ny, nf]
    x = traj[:,:Par['lb']  ] #[bs, lb, nx,ny,nf]
    y = traj[:, Par['lb']: ] #[bs , nt-lb, nx ,ny, nf]

    x = x.transpose(0,4,1,2,3) #[bs, nf, lb,nx,ny]
    y = y.transpose(0,4,1,2,3) #[bs, nf, nt-lb,nx,ny]

    print('x: ', x.shape)
    print('y: ', y.shape)
    print()

    return x.astype(np.float32), y.astype(np.float32)

def rollout(model, x, bs, Par):
    # x - [bs, nf, lb,nx,ny]

    NT = Par['nt']
    y_pred_ls = []
    # lf = 
    # nx = 64
    # ny = 64

    bs = bs
    end= bs
    for end in range(bs, x.shape[0]+1, bs):
        # print('end: ', end)
        start = end-bs
        out_ls = [x[start:end, :, :Par['lb']].to(device)]
        
        temp_x1 = x[start:end, :, -Par['lb']:] #[BS, nf, lb, nx,ny]
        while (len(out_ls)-1)*(Par['lf'])<NT:
            model.eval()
            with torch.no_grad():
                temp_x = temp_x1
                # print('temp_x: ', temp_x.shape)
                with autocast():
                    out = model(temp_x.to(device)) #[BS, nf, lf,nx,ny]
                out_ls.append(out.to(device))
                temp_out = torch.cat(out_ls, dim=2)
                temp_x1 = temp_out[:,:,-Par['lb']:] #[BS, nf, lb,nx,ny]
                                
        pred = torch.cat(out_ls, dim=2)[:, :, Par['lb']:NT] #[BS, nf, nt-lb, nx, ny]
        # print('pred: ', pred.shape)
        y_pred_ls.append(pred)

    y_pred = torch.cat(y_pred_ls, dim=0)#.reshape(1,-1,Par['nz'],Par['ny'],Par['nx'])
    # print('y_pred: ', y_pred.shape)

    return y_pred


In [6]:
temp  = "_sub.npy"
train = np.load("../data/train"+temp)

train.shape

(2080, 14, 128, 128, 3)

In [10]:
print("Loading Dataset ...")
debug = False

if debug:
    temp  = "_debug.npy"
    train = np.load("../data/train"+temp) #[BS, nt, nx, ny, nf]
    val   = np.load("../data/val"+temp)
    test  = np.load("../data/test"+temp)
else:
    temp  = "_sub.npy"
    train = np.load("../data/train"+temp)
    val   = np.load("../data/val"+temp)
    test  = np.load("../data/test"+temp)
print("Loaded Dataset")
print("Dataset type: ", train.dtype)

MEAN = np.load('../data/MEAN.npy').reshape(1,-1,1,1,1)
STD  = np.load('../data/STD.npy').reshape(1,-1,1,1,1)
MIN  = np.load('../data/MIN.npy').reshape(1,-1,1,1,1)
MAX  = np.load('../data/MAX.npy').reshape(1,-1,1,1,1)

print(f"MEAN: {MEAN.shape}\nSTD: {STD.shape}\nMIN: {MIN.shape}\nMAX: {MAX.shape}")

'''
change n_channels
'''

Par = {
       'DEVICE'          : device,
       'nt'              : train.shape[1],
       'nx'              : train.shape[2],
       'ny'              : train.shape[3],
       'nf'              : train.shape[4],
       'lb'              : 4,
       'lf'              : 1,
       'subsample_t'     : 1
       }

Par["temp"] = Par["nt"] - Par["lb"] - Par["lf"] + 1

print('\nTrain Dataset')
x_train,y_train = preprocess_train(train, Par)

print('\nfTrain Dataset')
x_ftrain,y_ftrain = preprocess(train, Par)
print('\nValidation Dataset')
x_val,y_val = preprocess(val, Par)
print('\nTest Dataset')
x_test,y_test = preprocess(test, Par)

# sys.exit()

Par.update(
       {
       'inp_ch'          : Par['nf']*Par['lb'],
       'out_ch'          : Par['nf']*Par['lf'],
       'n_channels'      : 32,
       'k'               : 3,
       'block_type_ls'  : ['Conv'],
       'inp_shift'       : torch.tensor(MEAN, dtype=DTYPE, device=device),
       'inp_scale'       : torch.tensor(STD, dtype=DTYPE, device=device),
       'out_shift'       : torch.tensor(MEAN, dtype=DTYPE, device=device),
       'out_scale'       : torch.tensor(STD, dtype=DTYPE, device=device),
       'out_shift_loss'  : 0,
       'out_scale_loss'  : 1,
       }
)


if debug:
    Par['num_epochs']  = 50 #500 #500
else:
    Par['num_epochs']  = 50 #4000

print('Par:\n', Par)

# Create custom datasets
x_train_tensor = torch.tensor(x_train, dtype=DTYPE)
y_train_tensor = torch.tensor(y_train, dtype=DTYPE)

x_ftrain_tensor = torch.tensor(x_ftrain, dtype=DTYPE)
y_ftrain_tensor = torch.tensor(y_ftrain, dtype=DTYPE)

x_val_tensor   = torch.tensor(x_val,   dtype=DTYPE)
y_val_tensor   = torch.tensor(y_val,   dtype=DTYPE)

x_test_tensor  = torch.tensor(x_test,  dtype=DTYPE)
y_test_tensor  = torch.tensor(y_test,  dtype=DTYPE)

train_dataset = YourDataset(x_train_tensor, y_train_tensor)
ftrain_dataset = YourDataset(x_ftrain_tensor, y_ftrain_tensor)
val_dataset = YourDataset(x_val_tensor, y_val_tensor)
test_dataset = YourDataset(x_test_tensor, y_test_tensor)

# Define data loaders
train_batch_size = 32
ftrain_batch_size = 208
val_batch_size   = x_val.shape[0]
test_batch_size  = x_test.shape[0]
train_loader = DataLoader(train_dataset, batch_size=train_batch_size, shuffle=True)
ftrain_loader = DataLoader(ftrain_dataset, batch_size=ftrain_batch_size)
val_loader = DataLoader(val_dataset, batch_size=val_batch_size)
test_loader = DataLoader(test_dataset, batch_size=test_batch_size)

Loading Dataset ...
Loaded Dataset
Dataset type:  float64
MEAN: (1, 3, 1, 1, 1)
STD: (1, 3, 1, 1, 1)
MIN: (1, 3, 1, 1, 1)
MAX: (1, 3, 1, 1, 1)

Train Dataset
x:  (20800, 3, 4, 128, 128)
y:  (20800, 3, 1, 128, 128)


fTrain Dataset
x:  (2080, 3, 4, 128, 128)
y:  (2080, 3, 10, 128, 128)


Validation Dataset
x:  (260, 3, 4, 128, 128)
y:  (260, 3, 10, 128, 128)


Test Dataset
x:  (260, 3, 4, 128, 128)
y:  (260, 3, 10, 128, 128)

Par:
 {'DEVICE': device(type='cuda'), 'nt': 14, 'nx': 128, 'ny': 128, 'nf': 3, 'lb': 4, 'lf': 1, 'subsample_t': 1, 'temp': 10, 'inp_ch': 12, 'out_ch': 3, 'n_channels': 32, 'k': 3, 'block_type_ls': ['Conv'], 'inp_shift': tensor([[[[[ 7.1653e-01]]],


         [[[ 4.6689e-10]]],


         [[[-3.1769e-08]]]]], device='cuda:0'), 'inp_scale': tensor([[[[[0.4421]]],


         [[[0.4395]]],


         [[[0.5088]]]]], device='cuda:0'), 'out_shift': tensor([[[[[ 7.1653e-01]]],


         [[[ 4.6689e-10]]],


         [[[-3.1769e-08]]]]], device='cuda:0'), 'out_scale': ten

In [11]:
# Initialize your Unet2D model
model = MnM(Par).to(device).to(DTYPE)
print(summary(model, input_size=(1,)+x_train.shape[1:]) )


print('Loading model ...')
path_model = 'models/best_model.pt'
model.load_state_dict(torch.load(path_model))



# Adjust the dimensions as per your model's input size
dummy_input = torch.tensor(torch.randn(1, Par['nf'], Par['lb'], Par['nx'],Par['ny']),   dtype=DTYPE, device=device)

# Profile the model
flops = torchprofile.profile_macs(model, dummy_input)
print(f"FLOPs: {flops}")

# Define loss function and optimizer
criterion = CustomLoss(Par)
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

Layer (type:depth-idx)                             Output Shape              Param #
MnM                                                [1, 3, 1, 128, 128]       --
├─Conv_Block: 1-1                                  [1, 32, 128, 128]         --
│    └─Sequential: 2-1                             [1, 32, 128, 128]         --
│    │    └─Conv2d: 3-1                            [1, 32, 128, 128]         416
│    │    └─GroupNorm: 3-2                         [1, 32, 128, 128]         64
│    │    └─GELU: 3-3                              [1, 32, 128, 128]         --
├─Enc_Block: 1-2                                   [1, 32, 128, 128]         --
│    └─Identity: 2-2                               [1, 32, 128, 128]         --
│    └─MnM_Block: 2-3                              [1, 32, 128, 128]         --
│    │    └─ModuleList: 3-4                        --                        9,312
│    │    └─Conv2d: 3-5                            [1, 32, 128, 128]         2,080
│    └─Conv_Block: 2-4      

/tmp/ipykernel_1008691/818682496.py:13: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  dummy_input = torch.tensor(torch.randn(1, Par['nf'], Par['lb'], Par['nx'],Par['ny']),   dtype=DTYPE, device=device)
/oscar/home/voommen/apps/torch_env/lib64/python3.9/site-packages/torchprofile/profile.py:22: UserWarning: No handlers found: "aten::reshape". Skipped.
  warnings.warn('No handlers found: "{}". Skipped.'.format(
/oscar/home/voommen/apps/torch_env/lib64/python3.9/site-packages/torchprofile/profile.py:22: UserWarning: No handlers found: "aten::_convolution_mode". Skipped.
  warnings.warn('No handlers found: "{}". Skipped.'.format(
/oscar/home/voommen/apps/torch_env/lib64/python3.9/site-packages/torchprofile/profile.py:22: UserWarning: No handlers found: "aten::gelu". Skipped.
  warnings.warn('No handlers found: "{}". Skipped.'.format(


FLOPs: 561119233


/oscar/home/voommen/apps/torch_env/lib64/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [12]:
y_true_ls = []
y_pred_ls = []

model.eval()
train_loss = 0.0
with torch.no_grad():
    for x, y_true in ftrain_loader:
        with autocast():
            y_pred = rollout(model, x, ftrain_batch_size, Par)
            loss   = criterion(y_pred, y_true.to(device), Par)
        train_loss += loss.item()
        y_true_ls.append(y_true.detach().cpu().numpy())
        y_pred_ls.append(y_pred.detach().cpu().numpy())

train_loss /= len(train_loader)
print(f"Train Loss: {train_loss:.4e}")

TRAIN_TRUE = np.concatenate(y_true_ls, axis=0)#.reshape(-1, Par['lf'], Par['nx'], Par['ny']).astype(np.float32)
TRAIN_PRED = np.concatenate(y_pred_ls, axis=0)#.reshape(-1, Par['lf'], Par['nx'], Par['ny']).astype(np.float32)

print(f"TRAIN_TRUE: {TRAIN_TRUE.shape}, DTYPE: {TRAIN_TRUE.dtype}")
print(f"TRAIN_PRED: {TRAIN_PRED.shape}, DTYPE: {TRAIN_PRED.dtype}")



y_true_ls = []
y_pred_ls = []

model.eval()
val_loss = 0.0
with torch.no_grad():
    for x, y_true in val_loader:
        with autocast():
            y_pred = rollout(model, x, val_batch_size, Par)
            loss   = criterion(y_pred, y_true.to(device), Par)
        val_loss += loss.item()
        y_true_ls.append(y_true.detach().cpu().numpy())
        y_pred_ls.append(y_pred.detach().cpu().numpy())

val_loss /= len(val_loader)
print(f"Val Loss: {val_loss:.4e}")

VAL_TRUE = np.concatenate(y_true_ls, axis=0)#.reshape(-1, Par['lf'], Par['nx'], Par['ny']).astype(np.float32)
VAL_PRED = np.concatenate(y_pred_ls, axis=0)#.reshape(-1, Par['lf'], Par['nx'], Par['ny']).astype(np.float32)

print(f"VAL_TRUE: {VAL_TRUE.shape}, DTYPE: {VAL_TRUE.dtype}")
print(f"VAL_PRED: {VAL_PRED.shape}, DTYPE: {VAL_PRED.dtype}")


x_true_ls = []
y_true_ls = []
y_pred_ls = []

model.eval()
test_loss = 0.0
with torch.no_grad():
    for x, y_true in test_loader:
        with autocast():
            y_pred = rollout(model, x, val_batch_size, Par)
            loss   = criterion(y_pred, y_true.to(device), Par)
        test_loss += loss.item()
        x_true_ls.append(x.detach().cpu().numpy())
        y_true_ls.append(y_true.detach().cpu().numpy())
        y_pred_ls.append(y_pred.detach().cpu().numpy())

test_loss /= len(test_loader)
print(f"Test Loss: {test_loss:.4e}")

TEST_X = np.concatenate(x_true_ls, axis=0)#.reshape(-1, Par['lb'], Par['nx'], Par['ny']).astype(np.float32)
TEST_TRUE = np.concatenate(y_true_ls, axis=0)#.reshape(-1, Par['lf'], Par['nx'], Par['ny']).astype(np.float32)
TEST_PRED = np.concatenate(y_pred_ls, axis=0)#.reshape(-1, Par['lf'], Par['nx'], Par['ny']).astype(np.float32)

print(f"TEST_X: {TEST_X.shape}, DTYPE: {TEST_X.dtype}")
print(f"TEST_TRUE: {TEST_TRUE.shape}, DTYPE: {TEST_TRUE.dtype}")
print(f"TEST_PRED: {TEST_PRED.shape}, DTYPE: {TEST_PRED.dtype}")

Train Loss: 5.7293e-03
TRAIN_TRUE: (2080, 3, 10, 128, 128), DTYPE: float32
TRAIN_PRED: (2080, 3, 10, 128, 128), DTYPE: float32
Val Loss: 5.9455e-01
VAL_TRUE: (260, 3, 10, 128, 128), DTYPE: float32
VAL_PRED: (260, 3, 10, 128, 128), DTYPE: float32
Test Loss: 5.6899e-01
TEST_X: (260, 3, 4, 128, 128), DTYPE: float32
TEST_TRUE: (260, 3, 10, 128, 128), DTYPE: float32
TEST_PRED: (260, 3, 10, 128, 128), DTYPE: float32


In [13]:
np.save("TRAIN_TRUE.npy", TRAIN_TRUE)
np.save("TRAIN_PRED.npy", TRAIN_PRED)

np.save("VAL_TRUE.npy", VAL_TRUE)
np.save("VAL_PRED.npy", VAL_PRED)

np.save("TEST_X.npy", TEST_X)
np.save("TEST_TRUE.npy", TEST_TRUE)
np.save("TEST_PRED.npy", TEST_PRED)